# Run a config on a Kaggle GPU

This notebook runs one or more configs from `configs/` on Kaggle (GPU T4 x2). Use it for the 32B Bob.

**Set up the notebook once:**
1. Settings: set Accelerator to **GPU T4 x2** and set **Internet on**.
2. Make the bundle (`scripts/make_bundle.py`). Upload it as a **private** dataset and attach it.
3. If the bundle has no encrypted GPQA, add the secret **HF_TOKEN** (a Hugging Face token with GPQA access).

**Start a run:** use `scripts/kaggle_push.py`. It fills in cell 1 and pushes a new version.

The bundle cache supplies all calls that were done before. Only new calls use the GPU.
Every 30 minutes the notebook writes `oversight_checkpoint.zip`. If a run stops at the 12-hour limit, unzip the checkpoint into the project and run again.

## 1. Settings

In [ ]:
# Config names in configs/ (without .yaml), run in order. Use one long config for each 12-hour session.
#   e.g. "kaggle_solve_x4_jev", "kaggle_notes_x4_jev", "kaggle_checks_32b"
CONFIGS = ["kaggle_solve_x4_jev"]
MAX_ITEMS = None      # e.g. 60 for a pilot on a subset of the same eval set (old conditions stay cached); None = all
GPQA_KEY = None       # filled in by scripts/kaggle_push.py at push time; never commit a real key

## 2. Find the bundle and set up the project

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

# Kaggle's image is built on Colab's (COLAB_RELEASE_TAG is set there too), and a /kaggle folder exists
# on Colab, so check Kaggle's own markers first: its run-type variable or an attached /kaggle/input.
_kaggle_input = Path("/kaggle/input")
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or (_kaggle_input.is_dir() and any(_kaggle_input.iterdir()))
ON_COLAB = not ON_KAGGLE
WORK = Path("/tmp/judge-oversight" if ON_KAGGLE else "/content/judge-oversight")  # not /kaggle/working: Kaggle publishes every file there
print("platform:", "Kaggle" if ON_KAGGLE else "Colab")

def find_bundle():
    # This platform's usual place first, then the other's, so a misdetected platform still finds it.
    roots = ["/kaggle/input", "/content", "/content/drive/MyDrive"]
    if not ON_KAGGLE:
        roots = roots[1:] + roots[:1]
    # The zip first (newest upload wins); an unpacked folder only on Kaggle, which unzips datasets itself.
    # Never pick WORK itself: it may hold an older extraction.
    for root in roots:
        if not Path(root).is_dir():
            continue
        zips = sorted(glob.glob(f"{root}/**/oversight_bundle*.zip", recursive=True), key=os.path.getmtime)
        if zips:
            return None, Path(zips[-1])
        for p in glob.glob(f"{root}/**/pyproject.toml", recursive=True):
            d = Path(p).parent
            if d.resolve() != WORK.resolve() and d.joinpath("src/oversight/data/gpqa.py").exists():
                return d, None
    raise FileNotFoundError("oversight_bundle not found: attach it as a dataset (Kaggle) or upload the zip to /content (Colab)")

# Always refresh the code from the bundle (overwrites old files, keeps any cache files already computed here).
folder, zpath = find_bundle()
print("bundle:", zpath or folder)
if folder:
    shutil.copytree(folder, WORK, dirs_exist_ok=True)
else:
    WORK.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(WORK.parent)
os.chdir(WORK)
assert Path("src/oversight/data/gpqa.py").exists(), "bundle is incomplete: src/oversight/data/gpqa.py missing"
print("project:", WORK, "| cached calls:", sum(1 for _ in Path("cache").rglob("*.json")))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!{sys.executable} -m pip install -q pydantic pyyaml httpx huggingface_hub cryptography

## 3. GPQA data

The bundle carries GPQA encrypted; with `GPQA_KEY` set it is decrypted locally and no Hugging Face secret is needed. Otherwise the `HF_TOKEN` secret downloads it.

In [ ]:
enc, csv_path = Path("data_enc/gpqa_diamond.csv.fernet"), Path("data/gpqa_diamond.csv")
if GPQA_KEY and enc.exists():
    from cryptography.fernet import Fernet
    csv_path.parent.mkdir(exist_ok=True)
    csv_path.write_bytes(Fernet(GPQA_KEY.encode()).decrypt(enc.read_bytes()))
    print("GPQA decrypted from the bundle:", csv_path)
else:
    try:
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        except ImportError:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        raise SystemExit(f"No GPQA source: set GPQA_KEY (encrypted bundle) or attach the HF_TOKEN secret ({e})")
    print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 4. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh

env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MAX_LOADED_MODELS="3", OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
import httpx
for _ in range(60):
    try:
        print("ollama", httpx.get("http://127.0.0.1:11434/api/version").json()["version"]); break
    except Exception:
        time.sleep(2)

## 5. Pull models

Alice (`qwen2.5:7b`) isn't pulled: her answers come from the cache.

In [ ]:
import yaml
needed = []
for name in CONFIGS:
    models = yaml.safe_load(open(f"configs/{name}.yaml"))["models"]
    needed += [spec["model"] for role, spec in models.items() if role != "alice"]  # Alice comes from the cache
for m in dict.fromkeys(needed):
    print("pulling", m, flush=True)
    subprocess.run(["ollama", "pull", m], env=env, check=True, stdout=subprocess.DEVNULL)
!ollama list

## 6. Run the sweep

Every 30 minutes a checkpoint (`oversight_checkpoint.zip`: results + cache) is written next to the outputs. If the session hits its time limit, download it and unzip it into the project: a re-run resumes from it.

In [ ]:
import threading

OUT_DIR = Path("/kaggle/working" if ON_KAGGLE else "/content")

def save_zip(path):
    tmp = path.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for folder in ["results", "cache"]:
            for f in Path(folder).rglob("*"):
                if f.is_file() and f.suffix != ".tmp":
                    z.write(f)
    tmp.replace(path)

stop_checkpoints = threading.Event()
def checkpoint_loop():
    while not stop_checkpoints.wait(30 * 60):
        save_zip(OUT_DIR / "oversight_checkpoint.zip")
        print(f"[checkpoint saved {time.strftime('%H:%M')}]", flush=True)
threading.Thread(target=checkpoint_loop, daemon=True).start()

for name in CONFIGS:
    cmd = [sys.executable, "-m", "oversight.run", "run", "--config", f"configs/{name}.yaml"]
    if MAX_ITEMS:
        cmd += ["--max-eval-items", str(MAX_ITEMS), "--out", f"results/{name}_pilot{MAX_ITEMS}"]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, env=dict(os.environ, PYTHONPATH="src"), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    print("exit code", proc.wait())

## 7. Results and download

In [ ]:
for p in sorted(Path("results").glob("kaggle_*/summary.md")):
    print("=" * 80, "\n", p, "\n"); print(p.read_text())

stop_checkpoints.set()
out = OUT_DIR / "oversight_outputs.zip"
save_zip(out)
print("download:", out, f"({out.stat().st_size / 1e6:.1f} MB)")